# ✂️ Auto Clipper - Google Colab GPU Backend Server

Notebook ini menjalankan backend **Auto Clipper** di lingkungan GPU Google Colab (T4 GPU Gratis) untuk akselerasi transkripsi Whisper, pemotongan video 9:16 (Face Tracking / Canvas), dan rendering FFmpeg NVENC.

---

### 1. Hubungkan Google Drive (Mount Drive)
Hubungkan Google Drive Anda untuk menyimpan riwayat proyek, video asli, database (`history.db`), dan klip hasil render secara permanen.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Pasang Sistem & Dependensi (FFmpeg & Cloudflared)
Memasang `ffmpeg` untuk pengolah video dan `cloudflared` untuk tunnel jaringan aman.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

### 3. Unduh Program Auto Clipper & Dependensi Python
Mengunduh repositori dan memasang seluruh library Python yang dibutuhkan.

In [ ]:
!git clone https://github.com/DhimasPH/auto-clipper.git /content/auto-clipper || true
%cd /content/auto-clipper
!pip install -q -r backend/requirements.txt uvicorn pyngrok

### 4A. Jalankan Backend (Pilihan A: Cloudflare Tunnel - Punya Domain Sendiri)
Masukkan **Cloudflare Tunnel Token** dan **API Secret Token (Password Rahasia)** pada formulir di bawah ini, lalu klik tombol **Play (▶️)**.

In [ ]:
#@title Jalankan Server via Cloudflare Tunnel
CLOUDFLARE_TUNNEL_TOKEN = "" #@param {type:"string"}
API_SECRET_TOKEN = "" #@param {type:"string"}

!python backend/colab_api.py --cloudflare-token "$CLOUDFLARE_TUNNEL_TOKEN" --api-token "$API_SECRET_TOKEN"

### 4B. Jalankan Backend (Pilihan B: Ngrok Tunnel - 100% Gratis Tanpa Domain)
Jika Anda tidak memiliki domain pribadi di Cloudflare, gunakan formulir di bawah ini dengan **Ngrok Authtoken** (Daftar gratis di ngrok.com).

In [ ]:
#@title Jalankan Server via Ngrok Tunnel
NGROK_AUTHTOKEN = "" #@param {type:"string"}
API_SECRET_TOKEN = "" #@param {type:"string"}

from pyngrok import ngrok
import os

if NGROK_AUTHTOKEN.strip():
    ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())

public_url = ngrok.connect(8000).public_url
print(f"\n=========================================")
print(f"🚀 PUBLIC BACKEND URL ANDA: {public_url}")
print(f"=========================================\n")

os.environ["API_SECRET_TOKEN"] = API_SECRET_TOKEN.strip()
os.environ["AUTO_CLIPPER_DEV_TOKEN"] = API_SECRET_TOKEN.strip()
os.environ["AUTO_CLIPPER_WEB_TOKEN"] = API_SECRET_TOKEN.strip()
os.environ["AUTO_CLIPPER_CLOUD_MODE"] = "1"
os.environ["AUTO_CLIPPER_WORKSPACE"] = "/content/drive/MyDrive/AutoClipperData"

!python -m uvicorn backend.main:app --host 0.0.0.0 --port 8000